# Imports

In [78]:
import pandas as pd
import numpy as np
import sklearn
from matplotlib import pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

In [79]:
df=pd.read_csv('project_excel_raw.csv')

In [80]:
df.drop('Timestamp', axis=1, inplace=True)

In [81]:
df.drop(df.columns[0], axis=1, inplace=True)

In [82]:
new_column_names = [
    'Age',
    'Gender',
    'Occupation',
    'Daily_device_hour',
    'Most_frequently_used_device',
    'Smartphone_check_frequency_wakeup',
    'Smartphone_check_frequency_bedtime',
    'Uses_tech_for_work_or_academic',
    'Social_media_frequency',
    'Daily_tv_streaming_hours',
    'Tech_for_relaxation_or_entertainment',
    'Most_frequent_online_content',
    'Emotions_associated_with_tech',
    'Mental_wellbeing_rating',
    'Emotional_symptoms_frequency',
    'Concentration_decision_difficulty_frequency',
    'Sleep_quality',
    'Changes_in_appetite_or_weight',
    'Physical_symptoms_frequency',
    'Energy_motivation_rating',
    'Interest_in_activities',
    'Stress_overwhelm_frequency',
    'Isolation_feeling',
    'Has_mental_health_support',
    'Mental_wellbeing_activities_frequency',
    'Digital_detox_taken',
    'Longest_digital_detox_duration',
    'Digital_detox_motivation',
    'Digital_detox_emotional_experience',
    'Digital_detox_goals_set',
    'Consider_regular_digital_detox',
    'Digital_detox_impact_on_tech_use',
    'Digital_detox_challenges',
    'Tech_stress_management',
    'Tech_stress_management_effectiveness',
    'Physical_activities_for_stress_management',
    'Outdoor_activities_to_reduce_tech_stress',
    'Tech_stress_coping_strategies',
]

df.columns = new_column_names

In [83]:
df.head()

,Age,Gender,Occupation,Daily_device_hour,Most_frequently_used_device,Smartphone_check_frequency_wakeup,Smartphone_check_frequency_bedtime,Uses_tech_for_work_or_academic,Social_media_frequency,Daily_tv_streaming_hours,...,Digital_detox_emotional_experience,Digital_detox_goals_set,Consider_regular_digital_detox,Digital_detox_impact_on_tech_use,Digital_detox_challenges,Tech_stress_management,Tech_stress_management_effectiveness,Physical_activities_for_stress_management,Outdoor_activities_to_reduce_tech_stress,Tech_stress_coping_strategies
0,23,Male,Student,5-6 hours,Smartphone,Always,Always,Yes,Multiple times a day,Less than 1 hour,...,No significant change,Yes,Yes,Used technology less,Difficulty staying disconnected,Set specific time limits for technology use,4,Frequently,No,I prefer to cope alone
1,23,Female,Student,More than 6 hours,Smartphone,Always,Always,Yes,Multiple times a day,Less than 1 hour,...,Relaxed and refreshed,Yes,Yes,No change,Social pressure to stay connected,Engage in outdoor activities,4,Occasionally,Yes,I prefer to cope alone
2,31,Male,Student,5-6 hours,Computer/Laptop,Always,Always,Yes,Multiple times a day,More than 6 hours,...,I have never practiced digital detox,No,No,I have never practiced digital detox,I have never practiced digital detox,I sleep,1,Never,No,I sleep very much
3,23,Male,Employed (full-time),5-6 hours,Smartphone,Always,Always,Yes,Multiple times a day,1-2 hours,...,I have never practiced digital detox,No,Yes,I have never practiced digital detox,I have never practiced digital detox,Take short breaks from screens,2,Rarely,No,I prefer to cope alone
4,18,Female,Student,More than 6 hours,Smartphone,Often,Often,Yes,Multiple times a day,Less than 1 hour,...,Relaxed and refreshed,Yes,Yes,No change,Difficulty staying disconnected,Talk to friends or family about technology-rel...,3,Rarely,Yes,I seek support from friends or family


# Train-Test-Split

In [84]:
from sklearn.model_selection import train_test_split
train_set, test_set = train_test_split(df, test_size=0.15,random_state=14)

In [85]:
train_set.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 536 entries, 364 to 619
Data columns (total 38 columns):
 #   Column                                       Non-Null Count  Dtype 
---  ------                                       --------------  ----- 
 0   Age                                          536 non-null    object
 1   Gender                                       536 non-null    object
 2   Occupation                                   536 non-null    object
 3   Daily_device_hour                            536 non-null    object
 4   Most_frequently_used_device                  536 non-null    object
 5   Smartphone_check_frequency_wakeup            536 non-null    object
 6   Smartphone_check_frequency_bedtime           536 non-null    object
 7   Uses_tech_for_work_or_academic               536 non-null    object
 8   Social_media_frequency                       536 non-null    object
 9   Daily_tv_streaming_hours                     536 non-null    object
 10  Tech_for_rel

# Data Pre-processing

1. Age

In [86]:
#Regular Expression used to extract digits from Age Column
train_set['Age']=train_set['Age'].str.extract('(\d+)', expand=False)
test_set['Age']=test_set['Age'].str.extract('(\d+)', expand=False)

In [87]:
train_set['Age']=train_set['Age'].astype('float64')
test_set['Age']=test_set['Age'].astype('float64')

In [88]:
# feature scaling age
from sklearn.preprocessing import MinMaxScaler
age_column_train = train_set['Age'].values.reshape(-1, 1)
age_column_test = test_set['Age'].values.reshape(-1, 1)
mms = MinMaxScaler()
train_set['Age'] = mms.fit_transform(age_column_train)
test_set['Age'] = mms.transform(age_column_test)

In [89]:
median_age = train_set['Age'].median()
print(median_age)

0.10465116279069764


In [90]:
train_set['Age'].fillna(median_age, inplace=True)
test_set['Age'].fillna(median_age, inplace=True)

2. Converting user typed attributes to "others"

In [91]:
form_options = [
    "News and information",
    "Entertainment (videos, movies, music)",
    "Educational content",
    "Social media updates",
    "Work-related content",
    "Smartphone",
    "Computer/Laptop",
    "Tablet",
    "Feeling overwhelmed or stressed",
    "Noticing negative effects on mental health",
    "Focusing on in-person relationships",
    "Improving productivity",
    "Curiosity or personal experiment",
    "Relaxed and refreshed",
    "Anxious or stressed",
    "No significant change",
    "Used technology less",
    "No change",
    "Used technology more mindfully",
    "FOMO (Fear of Missing Out)",
    "Difficulty staying disconnected",
    "Social pressure to stay connected",
    "Boredom",
    "None, it was easy and enjoyable",
    "I have never practiced digital detox",
    "Take short breaks from screens",
    "Set specific time limits for technology use",
    "Engage in outdoor activities",
    "Practice mindfulness or meditation",
    "Talk to friends or family about technology-related stress",
    "Unplug from technology for a designated period",
    "Seek professional help or counseling",
    "I prefer to cope alone",
    "I seek support from friends or family",
    "I participate in online communities or forums",
    "I attend social events or gatherings",
]

target_cols = [
    'Most_frequently_used_device',
    'Most_frequent_online_content',
    'Digital_detox_motivation',
    'Digital_detox_emotional_experience',
    'Digital_detox_impact_on_tech_use',
    'Digital_detox_challenges',
    'Tech_stress_management',
    'Tech_stress_coping_strategies',
]

In [92]:
for col in target_cols:
    for value in train_set[col]:
        if value not in form_options:
            train_set[col].replace(value,"Others",inplace=True)

In [93]:
for col in target_cols:
    for value in test_set[col]:
        if value not in form_options:
            test_set[col].replace(value,"Others",inplace=True)

# Y-Data (Pandas Profiling)

In [94]:
# # from pandas_profiling import ProfileReport
# from ydata_profiling import ProfileReport
# prof = ProfileReport(df)
# prof.to_file(output_file='output.html')

# Point System 3

In [98]:
# Strategy 1 : Ordinal Encoding of Targets
ordinal_cols = [
    'Mental_wellbeing_rating',
    'Emotional_symptoms_frequency',
    'Concentration_decision_difficulty_frequency',
    'Sleep_quality',
    'Energy_motivation_rating',
    'Interest_in_activities',
    'Stress_overwhelm_frequency',
    'Isolation_feeling',
    'Has_mental_health_support',
    'Physical_activities_for_stress_management',
]

In [112]:
display_cols = test_set[ordinal_cols].head()

In [113]:
display_cols

,Mental_wellbeing_rating,Emotional_symptoms_frequency,Concentration_decision_difficulty_frequency,Sleep_quality,Energy_motivation_rating,Interest_in_activities,Stress_overwhelm_frequency,Isolation_feeling,Has_mental_health_support,Physical_activities_for_stress_management
114,3,Occasionally,Occasionally,Good,Moderate,Have lost interest,Frequently,Yes,Yes,Occasionally
446,3,Occasionally,Rarely,Excellent,Low,Have lost interest,Occasionally,No,Yes,Frequently
284,5,Not at all,Rarely,Fair,Low,Still enjoy activities,Rarely,No,No,Frequently
568,1,Almost all the time,Frequently,Fair,Moderate,Still enjoy activities,Rarely,Yes,No,Never
249,3,Occasionally,Rarely,Good,Moderate,Still enjoy activities,Occasionally,Yes,No,Occasionally


In [99]:
train_set_ordinal_cat=train_set[ordinal_cols]
test_set_ordinal_cat=test_set[ordinal_cols]

In [101]:
from sklearn.preprocessing import OrdinalEncoder
ordinal_encoder = OrdinalEncoder()
train_set_ordinal=ordinal_encoder.fit_transform(train_set_ordinal_cat)
train_set_ordinal = pd.DataFrame(train_set_ordinal, columns=ordinal_encoder.get_feature_names_out(ordinal_cols))

In [102]:
test_set_ordinal=ordinal_encoder.fit_transform(test_set_ordinal_cat)
test_set_ordinal = pd.DataFrame(test_set_ordinal, columns=ordinal_encoder.get_feature_names_out(ordinal_cols))

In [104]:
test_set_ordinal.head()

,Mental_wellbeing_rating,Emotional_symptoms_frequency,Concentration_decision_difficulty_frequency,Sleep_quality,Energy_motivation_rating,Interest_in_activities,Stress_overwhelm_frequency,Isolation_feeling,Has_mental_health_support,Physical_activities_for_stress_management
0,2.0,3.0,3.0,2.0,2.0,0.0,1.0,1.0,1.0,3.0
1,2.0,3.0,4.0,0.0,1.0,0.0,2.0,0.0,1.0,1.0
2,4.0,2.0,4.0,1.0,1.0,1.0,3.0,0.0,0.0,1.0
3,0.0,0.0,1.0,1.0,2.0,1.0,3.0,1.0,0.0,2.0
4,2.0,3.0,4.0,2.0,2.0,1.0,2.0,1.0,0.0,3.0


In [107]:
attribute_mapping = ordinal_encoder.categories_

# Print the mapping for each attribute
for i, attribute_values in enumerate(attribute_mapping):
    attribute_name = ordinal_encoder.get_feature_names_out()[i]
    print(f"{attribute_name} -> {attribute_values}")

Mental_wellbeing_rating -> [1 2 3 4 5]
Emotional_symptoms_frequency -> ['Almost all the time' 'Frequently' 'Not at all' 'Occasionally']
Concentration_decision_difficulty_frequency -> ['Almost all the time' 'Frequently' 'Never' 'Occasionally' 'Rarely']
Sleep_quality -> ['Excellent' 'Fair' 'Good' 'Poor' 'Very Poor']
Energy_motivation_rating -> ['High' 'Low' 'Moderate' 'Very High' 'Very low']
Interest_in_activities -> ['Have lost interest' 'Still enjoy activities']
Stress_overwhelm_frequency -> ['Almost all the time' 'Frequently' 'Occasionally' 'Rarely']
Isolation_feeling -> ['No' 'Yes']
Has_mental_health_support -> ['No' 'Yes']
Physical_activities_for_stress_management -> ['Always' 'Frequently' 'Never' 'Occasionally' 'Rarely']
